# Hands-on 2: Urban Thermal Comfort Modeling

**Module 2 — Deep Learning Best Practices & Applications**  
CIMA Foundation PhD Course, 27 May 2026

In this notebook we build a neural network to predict a thermal comfort index (UTCI) from meteorological and urban variables.  
Same stack as Notebook 1: Zarr + xarray, PyTorch Lightning, OmegaConf, MLFlow.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meteocima/Monaco_DLCourse/blob/main/notebooks/02_urban_thermal_comfort.ipynb)

## 0. Setup

We use [**uv**](https://docs.astral.sh/uv/) as our Python package manager — it's a fast, modern replacement for pip + virtualenv.

| Context | Command | What it does |
|---------|---------|-------------|
| **New project** | `uv init` | Scaffold a project with `pyproject.toml` |
| **Add a dependency** | `uv add torch xarray` | Add packages to `pyproject.toml` and install them |
| **Install everything** | `uv sync` | Create a `.venv` and install from the lockfile |
| **Run a command** | `uv run jupyter lab` | Run inside the managed `.venv` |
| **Colab / system Python** | `uv pip install --system -r requirements.txt` | Install without a `.venv` (for environments you don't manage) |

On **Colab** there is no `.venv` — Google provides the Python environment — so we use `uv pip install --system`.

In [ ]:
import os

# On Colab: clone the repo (or pull latest) and install dependencies
if "COLAB_RELEASE_TAG" in os.environ:
    if os.path.exists("/content/Monaco_DLCourse"):
        !git -C /content/Monaco_DLCourse pull -q
    else:
        !git clone https://github.com/meteocima/Monaco_DLCourse.git /content/Monaco_DLCourse
    %cd /content/Monaco_DLCourse/notebooks
    !pip install -q uv
    !uv pip install --system -q -r ../requirements.txt

In [ ]:
from __future__ import annotations

from typing import Any

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import xarray as xr
import zarr
from omegaconf import DictConfig, OmegaConf
from torch.utils.data import DataLoader, Dataset

print(f"PyTorch version: {torch.__version__}")
print(f"Lightning version: {pl.__version__}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print(
        "\n⚠️  No GPU detected!\n"
        "   Go to Runtime → Change runtime type → T4 GPU\n"
        "   Then re-run this cell.\n"
        "   Training will still work on CPU, but will be slower."
    )

## 1. Configuration with Hydra / OmegaConf

In production, Hydra loads YAML config files and composes them automatically via its `@hydra.main` decorator.  In a notebook we use **OmegaConf** (Hydra's config backend) to load the same YAML files directly — giving us structured, typed configuration without needing the CLI.

The `load_config()` function below loads the YAML and lets you **override any value** via keyword arguments — the same pattern Hydra uses on the command line (`max_epochs=50`), but adapted for notebooks.

In [ ]:
def load_config(
    config_path: str = "../configs/urban_thermal.yaml",
    **overrides: Any,
) -> DictConfig:
    """Load a YAML config and apply keyword overrides.

    Flat shorthand keys (e.g. ``max_epochs=50``, ``batch_size=64``)
    are mapped to their nested config paths automatically via the
    ``SHORTCUTS`` table.

    Args:
        config_path: Path to the YAML configuration file.
        **overrides: Keyword overrides forwarded to the config.

    Returns:
        Merged OmegaConf ``DictConfig``.
    """
    cfg = OmegaConf.load(config_path)

    SHORTCUTS: dict[str, str] = {
        "max_epochs": "training.max_epochs",
        "learning_rate": "training.learning_rate",
        "batch_size": "data.batch_size",
        "hidden_sizes": "model.hidden_sizes",
        "dropout": "model.dropout",
        "time_start": "data.time_start",
        "time_end": "data.time_end",
        "train_split": "data.train_split",
    }
    for key, value in overrides.items():
        path = SHORTCUTS.get(key, key)
        OmegaConf.update(cfg, path, value)

    return cfg


cfg = load_config()
print(OmegaConf.to_yaml(cfg))

## 2. Data: ERA5 Urban Climate Variables

We load **real ERA5 reanalysis data** from the [ARCO-ERA5](https://github.com/google-research/arco-era5) cloud Zarr store — no download needed!  The store contains the full ERA5 archive at 0.25° resolution, hosted on Google Cloud Storage.  **Zarr** is a chunked, compressed array format optimised for cloud storage: `xr.open_zarr()` reads only the *metadata* — actual data is fetched lazily when `.compute()` is called.

### 📥 Raw ERA5 Variables

| Variable | ERA5 name | Unit |
|----------|-----------|------|
| 2-m air temperature | `2m_temperature` | K |
| 2-m dewpoint temperature | `2m_dewpoint_temperature` | K |
| 10-m wind (u-component) | `10m_u_component_of_wind` | m s⁻¹ |
| 10-m wind (v-component) | `10m_v_component_of_wind` | m s⁻¹ |
| Surface downward solar radiation | `surface_solar_radiation_downwards` | J m⁻² |
| Mean sea level pressure | `mean_sea_level_pressure` | Pa |

### 🔧 Derived Features

From these we engineer five physical predictors:

| Feature | Formula | Unit |
|---------|---------|------|
| $T_{2m}$ | `2m_temperature` − 273.15 | °C |
| $T_d$ | `2m_dewpoint_temperature` − 273.15 | °C |
| $v_{10}$ | $\sqrt{u_{10}^2 + v_{10}^2}$ | m s⁻¹ |
| $R_s$ | `surface_solar_radiation_downwards` (clipped ≥ 0) | J m⁻² |
| $P_{\text{msl}}$ | `mean_sea_level_pressure` / 100 | hPa |

### 🎯 Target: Simplified UTCI

The [Universal Thermal Climate Index](https://utci.org/) integrates temperature, radiation, humidity, and wind into a single "feels-like" temperature.  Computing the full UTCI requires a multi-node human body model; here we use a **simplified approximation** that captures the main physical dependencies:

$$\text{UTCI} \approx 0.7\,T_{2m} + 0.2\,\frac{R_s}{100} - 0.5\,v_{10} - 0.3\,\Delta_h + 0.01\,T_{2m}\,\frac{R_s}{100}$$

where $\Delta_h = 0.5\,(T_{2m} - T_d)$ is a humidity-depression term.

**Physical interpretation of each term:**

- $0.7\,T_{2m}$ — **air temperature dominates** perceived warmth
- $0.2\,R_s / 100$ — **solar radiation** raises thermal stress (hot sun feels hotter)
- $-0.5\,v_{10}$ — **wind chill** effect: wind increases convective heat loss
- $-0.3\,\Delta_h$ — **dry air** (large $T_{2m} - T_d$) enhances evaporative cooling
- $0.01\,T_{2m}\,R_s / 100$ — **interaction**: radiation matters *more* when it's already hot

> **Note:** Opening the ARCO-ERA5 metadata may take ~30–60 seconds on first access.

In [ ]:
def load_data(cfg: DictConfig) -> dict[str, Any]:
    """Load ERA5 data for configured cities and derive features + UTCI.

    1. Opens the ARCO-ERA5 Zarr store lazily (metadata only).
    2. For each city in ``cfg.data.cities``, selects the nearest grid
       point and downloads the time range ``[time_start, time_end]``.
    3. Derives five physical features and the simplified UTCI target.

    Args:
        cfg: Configuration with ``data.zarr_url``, ``data.cities``,
            ``data.time_start``, and ``data.time_end``.

    Returns:
        Dict with keys ``features`` (N×5 float32 array),
        ``utci`` (N float32 array), ``feature_names`` (list[str]),
        and ``df`` (full pandas DataFrame).
    """
    surface_vars = [
        "2m_temperature",
        "2m_dewpoint_temperature",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "surface_solar_radiation_downwards",
        "mean_sea_level_pressure",
    ]

    # Open the ARCO-ERA5 Zarr store (lazy — only metadata is read)
    print("Opening ARCO-ERA5 store (this may take ~30-60s on first access)...")
    ds_era5 = xr.open_zarr(
        cfg.data.zarr_url,
        chunks=None,
        storage_options=dict(token="anon"),
    )
    print(f"Store opened. Available variables: {len(ds_era5.data_vars)}")

    # Extract surface variables for each city
    records: list[pd.DataFrame] = []
    for city, coords in cfg.data.cities.items():
        print(f"Loading {city} ({coords.lat}°N, {coords.lon}°E)...")
        ds_city = (
            ds_era5[surface_vars]
            .sel(
                latitude=coords.lat,
                longitude=coords.lon,
                method="nearest",
            )
            .sel(time=slice(cfg.data.time_start, cfg.data.time_end))
            .compute()
        )
        df = ds_city.to_dataframe().reset_index()
        df["city"] = city
        records.append(df)

    df_all = pd.concat(records, ignore_index=True).dropna()
    print(f"\nTotal records: {len(df_all)}")

    # Derive features
    df_all["t2m_C"] = df_all["2m_temperature"] - 273.15
    df_all["d2m_C"] = df_all["2m_dewpoint_temperature"] - 273.15
    df_all["wind_speed"] = np.sqrt(
        df_all["10m_u_component_of_wind"] ** 2
        + df_all["10m_v_component_of_wind"] ** 2
    )
    df_all["solar_rad"] = df_all[
        "surface_solar_radiation_downwards"
    ].clip(lower=0)
    df_all["mslp_hPa"] = df_all["mean_sea_level_pressure"] / 100

    # Simplified UTCI approximation
    humidity_effect = 0.5 * (df_all["t2m_C"] - df_all["d2m_C"])
    df_all["utci"] = (
        0.7 * df_all["t2m_C"]
        + 0.2 * df_all["solar_rad"] / 100
        - 0.5 * df_all["wind_speed"]
        - 0.3 * humidity_effect
        + 0.01 * df_all["t2m_C"] * df_all["solar_rad"] / 100
    ).astype(np.float32)

    feature_names = [
        "t2m_C", "d2m_C", "wind_speed", "solar_rad", "mslp_hPa",
    ]
    features = df_all[feature_names].values.astype(np.float32)
    utci = df_all["utci"].values.astype(np.float32)

    print(f"Features shape: {features.shape}")
    print(f"Target (UTCI) shape: {utci.shape}")
    print(f"UTCI range: [{utci.min():.1f}, {utci.max():.1f}] °C")

    return {
        "features": features,
        "utci": utci,
        "feature_names": feature_names,
        "df": df_all,
    }


data = load_data(cfg)

### 📊 Visualising the raw data

Before training, it's important to inspect the data.  The scatter plots below show each feature against the UTCI target.  This helps you:

- **Spot non-linear relationships** that justify using a neural network over simple linear regression
- **Identify outliers** or data-quality issues
- **Get a feel for feature relevance** — features with a clear trend against UTCI are likely to be important predictors

In [ ]:
def plot_raw_data(data: dict[str, Any]) -> None:
    """Scatter-plot each feature against the UTCI target.

    Produces one subplot per feature, coloured by UTCI value.

    Args:
        data: Dict returned by :func:`load_data` with keys
            ``features``, ``utci``, and ``feature_names``.
    """
    features: np.ndarray = data["features"]
    utci: np.ndarray = data["utci"]
    feature_names: list[str] = data["feature_names"]

    n_feat = len(feature_names)
    ncols = 3
    nrows = (n_feat + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    for i, (name, ax) in enumerate(zip(feature_names, axes.flat)):
        ax.scatter(
            features[:, i], utci,
            alpha=0.1, s=5, c=utci, cmap="RdYlBu_r",
        )
        ax.set_xlabel(name)
        ax.set_ylabel("UTCI (°C)")
        ax.set_title(f"{name} vs UTCI")

    for j in range(n_feat, nrows * ncols):
        axes.flat[j].set_visible(False)

    plt.suptitle("Feature–Target Relationships", fontsize=14)
    plt.tight_layout()
    plt.show()


plot_raw_data(data)

## 3. PyTorch Dataset & DataModule

### Why separate Dataset from Model?

PyTorch follows a **separation of concerns** pattern: the `Dataset` handles data access (*what is sample i?*), the `DataLoader` handles batching and shuffling, and the `Model` handles the forward pass and loss.  This makes each component reusable and independently testable.

### The `__getitem__` pattern

Each sample returned by `__getitem__(i)` is a tuple `(input, target)`:

```
features  (5,) ──────────→ input  (5,)    ← 5 meteorological predictors
utci      (1,) ──────────→ target (1,)    ← UTCI "feels-like" temperature
```

### Z-Score Normalisation

Unlike Notebook 1 (which uses min-max scaling), here we apply **Z-score** (standard) normalisation — more common for tabular data:

$$x_{\text{norm}} = \frac{x - \mu}{\sigma}$$

This centres each feature at zero with unit variance, which helps gradient-based optimisers converge faster — all features contribute on a comparable scale regardless of their original units (°C vs hPa vs J m⁻²).

> ⚠️ **Data leakage note:** in this notebook the normalisation statistics ($\mu$, $\sigma$) are computed on the full dataset.  This is acceptable because the UTCI target is a *synthetic* formula (not a real measurement), so there is no information leakage. In production, always compute stats on the training split only.

### Chronological Split

Samples are split into train / validation by **date order**, not randomly.  Weather records on consecutive hours are highly correlated (temporal autocorrelation), so a random split would leak near-duplicate samples across sets, inflating validation scores.

### 🔀 Shuffling & Batch Size

- **Training loader**: shuffled — breaks temporal ordering so consecutive batches are decorrelated, reducing gradient noise
- **Validation loader**: *not* shuffled — ensures reproducible metric computation

Training uses **mini-batch gradient descent**: instead of computing the loss over the entire dataset (slow) or a single sample (noisy), we average over a batch.  Larger batches give smoother gradients but need more GPU memory.

In [ ]:
class UrbanClimateDataset(Dataset):
    """Wraps pre-normalised feature/target arrays as a PyTorch Dataset.

    Each sample is a ``(features, target)`` tuple of float32 tensors.
    """

    def __init__(
        self,
        features: np.ndarray,
        targets: np.ndarray,
    ) -> None:
        self.features = torch.from_numpy(features)
        self.targets = torch.from_numpy(targets).unsqueeze(1)

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(
        self, idx: int,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        return self.features[idx], self.targets[idx]


class UrbanClimateDataModule(pl.LightningDataModule):
    """Lightning DataModule for the urban thermal comfort task.

    Handles Z-score normalisation and chronological train/val splitting.
    Normalisation statistics (``feat_mean``, ``feat_std``,
    ``target_mean``, ``target_std``) are stored as attributes so that
    predictions can be denormalised later.
    """

    def __init__(
        self,
        features: np.ndarray,
        targets: np.ndarray,
        cfg: DictConfig,
    ) -> None:
        super().__init__()
        self.features = features
        self.targets = targets
        self.cfg = cfg
        # Compute normalisation stats on full data
        self.feat_mean: np.ndarray = features.mean(axis=0)
        self.feat_std: np.ndarray = features.std(axis=0)
        self.target_mean: float = float(targets.mean())
        self.target_std: float = float(targets.std())

    def setup(self, stage: str | None = None) -> None:
        """Normalise and split into train / val datasets."""
        feat_norm = (self.features - self.feat_mean) / self.feat_std
        tgt_norm = (
            (self.targets - self.target_mean) / self.target_std
        )

        n = len(feat_norm)
        n_train = int(n * self.cfg.data.train_split)
        self.train_ds = UrbanClimateDataset(
            feat_norm[:n_train], tgt_norm[:n_train],
        )
        self.val_ds = UrbanClimateDataset(
            feat_norm[n_train:], tgt_norm[n_train:],
        )

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_ds,
            batch_size=self.cfg.data.batch_size,
            shuffle=True,
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self.val_ds,
            batch_size=self.cfg.data.batch_size,
        )


def create_datamodule(
    data: dict[str, Any],
    cfg: DictConfig,
) -> UrbanClimateDataModule:
    """Create, set up, and return the DataModule from loaded data.

    Args:
        data: Dict returned by :func:`load_data`.
        cfg: Configuration with ``data.train_split`` and
            ``data.batch_size``.

    Returns:
        Ready-to-use :class:`UrbanClimateDataModule`.
    """
    dm = UrbanClimateDataModule(data["features"], data["utci"], cfg)
    dm.setup()
    print(f"Train: {len(dm.train_ds)}, Val: {len(dm.val_ds)}")
    return dm


dm = create_datamodule(data, cfg)

## 4. Model: MLP for Thermal Comfort Prediction

A **Multi-Layer Perceptron (MLP)** — the simplest deep-learning architecture.  Unlike the CNN in Notebook 1 (which exploits *spatial* structure), the MLP treats each sample as a flat vector and learns through a stack of fully connected (dense) layers.

### Architecture

```
Input (5,) → [Linear → ReLU → Dropout] × N → Linear → Output (1,)
```

### Dense (Linear) Layer

Each layer applies a learned weight matrix and bias:

$$\mathbf{h} = \sigma\!\left(\mathbf{W}\mathbf{x} + \mathbf{b}\right)$$

where $\mathbf{x} \in \mathbb{R}^{d_{\text{in}}}$, $\mathbf{W} \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$, $\mathbf{b} \in \mathbb{R}^{d_{\text{out}}}$, and $\sigma$ is a non-linear activation.

### Activation Function — ReLU

$$\text{ReLU}(x) = \max(0, x)$$

Non-linear activations are essential: without them, stacking multiple linear layers would collapse into a *single* linear transformation ($\mathbf{W}_2\mathbf{W}_1 = \mathbf{W}'$), and the network could not learn complex non-linear relationships like the UTCI formula.

### Dropout — Regularisation

$$\text{Dropout}(x_i) = \begin{cases} 0 & \text{with probability } p \\ x_i / (1-p) & \text{otherwise}\end{cases}$$

During training, dropout **randomly zeroes** a fraction $p$ of neurons at each forward pass.  This prevents *co-adaptation*: neurons cannot rely on specific partners, forcing the network to learn more robust features.  At inference time dropout is disabled (all neurons active).

### 🔌 Injectable Components

The model accepts optional overrides for **loss function**, **optimizer**, and **activation** — swap them in the Playground to experiment:

| Component | Default | Alternatives to try |
|-----------|---------|---------------------|
| Loss | `nn.MSELoss()` | `nn.L1Loss()`, `nn.HuberLoss()` |
| Optimizer | `torch.optim.Adam` | `AdamW`, `SGD` |
| Activation | `nn.ReLU` | `nn.GELU`, `nn.SiLU`, `nn.LeakyReLU` |

> **Note:** Unlike the CNN in Notebook 1, this MLP has **no residual connection** — it learns the full $\text{features} \to \text{UTCI}$ mapping directly, rather than predicting a correction to an existing forecast.

In [ ]:
class ThermalComfortMLP(pl.LightningModule):
    """MLP for predicting UTCI from meteorological features.

    The network is a configurable stack of
    ``[Linear → activation → Dropout]`` blocks followed by a final
    ``Linear(→ 1)`` output layer.  Loss function, optimizer, and
    activation are injectable for easy experimentation.
    """

    def __init__(
        self,
        cfg: DictConfig,
        n_features: int,
        loss_fn: nn.Module | None = None,
        optimizer_cls: type[torch.optim.Optimizer] | None = None,
        activation_cls: type[nn.Module] | None = None,
    ) -> None:
        super().__init__()
        self.save_hyperparameters(
            ignore=["loss_fn", "optimizer_cls", "activation_cls"],
        )
        self.cfg = cfg
        self.loss_fn = loss_fn or nn.MSELoss()
        self.optimizer_cls = optimizer_cls or torch.optim.Adam
        activation_cls = activation_cls or nn.ReLU

        layers: list[nn.Module] = []
        in_size = n_features
        for hidden_size in cfg.model.hidden_sizes:
            layers.extend([
                nn.Linear(in_size, hidden_size),
                activation_cls(),
                nn.Dropout(cfg.model.dropout),
            ])
            in_size = hidden_size
        layers.append(nn.Linear(in_size, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass: features → predicted UTCI (normalised)."""
        return self.net(x)

    def training_step(
        self,
        batch: tuple[torch.Tensor, torch.Tensor],
        batch_idx: int,
    ) -> torch.Tensor:
        """Compute training loss and log it."""
        x, y = batch
        pred = self(x)
        loss = self.loss_fn(pred, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(
        self,
        batch: tuple[torch.Tensor, torch.Tensor],
        batch_idx: int,
    ) -> torch.Tensor:
        """Compute validation loss and MAE, log both."""
        x, y = batch
        pred = self(x)
        loss = self.loss_fn(pred, y)
        mae = F.l1_loss(pred, y)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_mae", mae)
        return loss

    def configure_optimizers(self) -> torch.optim.Optimizer:
        """Return the configured optimizer."""
        return self.optimizer_cls(
            self.parameters(), lr=self.cfg.training.learning_rate,
        )


def create_model(
    cfg: DictConfig,
    n_features: int,
    model_cls: type[pl.LightningModule] = ThermalComfortMLP,
    **model_kwargs: Any,
) -> pl.LightningModule:
    """Instantiate the MLP model from config.

    Args:
        cfg: Configuration with ``model.*`` keys.
        n_features: Number of input features.
        model_cls: Lightning module class to instantiate.
        **model_kwargs: Extra kwargs forwarded to the constructor
            (e.g. ``loss_fn``, ``optimizer_cls``, ``activation_cls``).

    Returns:
        Initialised Lightning module.
    """
    model = model_cls(cfg, n_features=n_features, **model_kwargs)
    print(model)
    return model


model = create_model(cfg, n_features=len(data["feature_names"]))

## 5. Training with Lightning + MLFlow Tracking

Lightning's `Trainer` handles the training loop, GPU placement, and logging.  We wrap the run in an **MLFlow context** so that every hyperparameter and metric is tracked automatically.  The `train()` function returns both the trainer and a dict of final metrics converted back to **°C** using the normalisation scale factor.

### 📉 Loss Function — MSE

We minimise the **Mean Squared Error** between predictions and truth:

$$\mathcal{L}_{\text{MSE}} = \frac{1}{N}\sum_{i=1}^{N}\left(y_i - \hat{y}_i\right)^2$$

Squaring penalises large errors disproportionately — a 10 °C error contributes 100× more than a 1 °C error.  Alternatives include **MAE** (L1 loss, more robust to outliers) and **Huber loss** (blends MSE for small errors with MAE for large ones).

### ⚡ Optimizer — Adam

**Adam** (Adaptive Moment Estimation) maintains per-parameter learning rates using running estimates of gradient mean and variance.  Intuition: it adapts step sizes per dimension — parameters with consistently large gradients take smaller steps, and vice versa.  This makes it far less sensitive to the initial learning rate than plain SGD.

### 🎛️ Learning Rate

The learning rate (default `1e-3`) controls step size in parameter space:
- **Too high** → the optimiser overshoots minima, loss oscillates or diverges
- **Too low** → convergence is painfully slow, training may get stuck in poor local minima

### 🔁 Epochs

One **epoch** = one complete pass through all training samples.  With 30 epochs, the model sees every sample 30 times, refining its parameters each pass.

### 📊 Validation Monitoring

After each epoch the trainer computes the loss on the held-out **validation set**.  This reveals overfitting: if training loss keeps decreasing but validation loss starts increasing, the model is memorising training data rather than learning generalisable patterns.

### 🔬 MLFlow Integration

Each call to `train()` creates an MLFlow *run* inside the configured *experiment*.  The run logs:
1. **All config parameters** (via `mlflow.log_params`)
2. **Final validation metrics** in physical units (°C)
3. A **`run_id`** returned to the caller so that evaluation metrics can be appended to the same run later

In [ ]:
def train(
    model: pl.LightningModule,
    datamodule: UrbanClimateDataModule,
    cfg: DictConfig,
    target_std: float,
    run_name: str = "mlp_thermal_comfort",
) -> tuple[pl.Trainer, dict[str, float], str]:
    """Train the model with Lightning and log to MLFlow.

    Args:
        model: The MLP to train.
        datamodule: Lightning DataModule (already set up).
        cfg: Configuration with ``training.*`` and ``mlflow.*`` keys.
        target_std: Standard deviation of the UTCI target, used to
            convert normalised metrics back to °C.
        run_name: Name for the MLflow run (shown in the runs table).

    Returns:
        Tuple of ``(trainer, metrics, run_id)`` where *metrics*
        contains ``val_rmse_utci`` and ``val_mae_utci`` in °C, and
        *run_id* is the MLflow run ID for logging additional metrics
        later.
    """
    mlflow.set_experiment(cfg.mlflow.experiment_name)

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(
            OmegaConf.to_container(cfg, resolve=True),
        )

        trainer = pl.Trainer(
            max_epochs=cfg.training.max_epochs,
            accelerator=cfg.training.accelerator,
            enable_progress_bar=True,
            log_every_n_steps=5,
        )
        trainer.fit(model, datamodule)

        metrics: dict[str, float] = {}
        val_loss = trainer.callback_metrics.get("val_loss")
        val_mae = trainer.callback_metrics.get("val_mae")
        if val_loss is not None:
            metrics["val_rmse_utci"] = float(
                val_loss.item() ** 0.5 * target_std,
            )
            mlflow.log_metric(
                "final_val_rmse_utci", metrics["val_rmse_utci"],
            )
        if val_mae is not None:
            metrics["val_mae_utci"] = float(
                val_mae.item() * target_std,
            )
            mlflow.log_metric(
                "final_val_mae_utci", metrics["val_mae_utci"],
            )

    rmse = metrics.get("val_rmse_utci")
    mae = metrics.get("val_mae_utci")
    if rmse is not None:
        print(f"\nFinal val RMSE: {rmse:.4f} °C")
    if mae is not None:
        print(f"Final val MAE:  {mae:.4f} °C")

    return trainer, metrics, run.info.run_id


trainer, metrics, run_id = train(model, dm, cfg, dm.target_std)

In [ ]:
def show_mlflow_runs(cfg: DictConfig) -> None:
    """Display a summary table of MLflow runs for the experiment.

    Queries the MLflow backend for all runs in
    ``cfg.mlflow.experiment_name`` and renders a tidy pandas
    DataFrame inline (no need for ``mlflow ui``).

    Args:
        cfg: Configuration with ``mlflow.experiment_name``.
    """
    from IPython.display import display

    experiment = mlflow.get_experiment_by_name(
        cfg.mlflow.experiment_name,
    )
    if experiment is None:
        print("No MLflow experiment found.")
        return

    df = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time DESC"],
    )
    if df.empty:
        print("No runs logged yet.")
        return

    # Build a tidy summary
    cols: dict[str, Any] = {}
    cols["Run Name"] = df["tags.mlflow.runName"]
    cols["Status"] = df["status"]
    cols["Start"] = pd.to_datetime(
        df["start_time"],
    ).dt.strftime("%H:%M:%S")

    # Include any metrics columns present
    for col in sorted(df.columns):
        if col.startswith("metrics."):
            short = col.replace("metrics.", "")
            cols[short] = df[col]

    # Key hyperparameters
    for key in [
        "params.training.max_epochs",
        "params.training.learning_rate",
        "params.model.hidden_sizes",
        "params.model.dropout",
    ]:
        if key in df.columns:
            short = key.replace("params.", "")
            cols[short] = df[key]

    summary = pd.DataFrame(cols)
    display(summary)

## 6. Evaluation & Visualization

The `predict()` function runs inference on the validation set and **denormalises** all outputs back to **°C** so that metrics are physically meaningful.  `evaluate()` then plots a prediction scatter and an error histogram.

### Denormalisation

To convert predictions back to UTCI °C we invert the Z-score scaling:

$$\hat{y}_{\text{UTCI}} = \hat{y}_{\text{norm}} \cdot \sigma_y + \mu_y$$

Reporting metrics in physical units (°C) makes them directly interpretable.

### 📐 Metrics

**RMSE** — Root Mean Squared Error (same units as the variable, penalises large errors):

$$\text{RMSE} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}\left(y_i - \hat{y}_i\right)^2}$$

**MAE** — Mean Absolute Error (linear penalty, more robust to outliers):

$$\text{MAE} = \frac{1}{N}\sum_{i=1}^{N}\left|y_i - \hat{y}_i\right|$$

**R²** — Coefficient of Determination (fraction of variance explained by the model):

$$R^2 = 1 - \frac{\sum_{i=1}^{N}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{N}(y_i - \bar{y})^2}$$

$R^2 = 1$ means perfect predictions; $R^2 = 0$ means the model is no better than always predicting the mean.

### 📊 Interpreting the Plots

- **Scatter plot** (predicted vs true): points close to the red diagonal = good predictions; systematic deviations reveal bias
- **Error histogram**: centred at zero = unbiased; wide spread = high variance

### 🔀 Permutation Feature Importance

To understand *which features matter most*, we **shuffle** one feature column at a time and re-run inference.  If shuffling a feature increases the error a lot, that feature was important for accurate predictions.  The importance score is the ratio $\text{MSE}_{\text{shuffled}} / \text{MSE}_{\text{baseline}}$ — values near 1.0 mean the feature doesn't matter; high values mean it's critical.

In [ ]:
def predict(
    model: pl.LightningModule,
    datamodule: UrbanClimateDataModule,
) -> dict[str, np.ndarray]:
    """Run inference on the validation set and denormalise to °C.

    Args:
        model: Trained MLP.
        datamodule: DataModule with ``val_ds`` populated and
            normalisation stats (``target_mean``, ``target_std``).

    Returns:
        Dict with ``pred_utci`` and ``true_utci`` (both in °C),
        and ``val_features`` (normalised tensors, kept for feature
        importance analysis).
    """
    model.eval()
    val_features: torch.Tensor = datamodule.val_ds.features
    val_targets: torch.Tensor = datamodule.val_ds.targets

    with torch.no_grad():
        val_pred = model(val_features)

    pred_utci: np.ndarray = (
        val_pred.numpy().squeeze()
        * datamodule.target_std
        + datamodule.target_mean
    )
    true_utci: np.ndarray = (
        val_targets.numpy().squeeze()
        * datamodule.target_std
        + datamodule.target_mean
    )

    return {
        "pred_utci": pred_utci,
        "true_utci": true_utci,
        "val_features": val_features,
    }


predictions = predict(model, dm)

In [ ]:
def evaluate(
    predictions: dict[str, np.ndarray],
) -> dict[str, float]:
    """Plot predicted vs true UTCI and error distribution.

    Args:
        predictions: Dict returned by :func:`predict` with
            ``pred_utci`` and ``true_utci`` (both in °C).

    Returns:
        Dict of evaluation metrics: ``eval_rmse_utci``,
        ``eval_mae_utci``, ``eval_r2``.
    """
    pred: np.ndarray = predictions["pred_utci"]
    true: np.ndarray = predictions["true_utci"]
    errors: np.ndarray = pred - true

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Prediction scatter
    axes[0].scatter(true, pred, alpha=0.2, s=5)
    lims = [
        min(true.min(), pred.min()),
        max(true.max(), pred.max()),
    ]
    axes[0].plot(lims, lims, "r--", lw=2, label="Perfect prediction")
    axes[0].set_xlabel("True UTCI (°C)")
    axes[0].set_ylabel("Predicted UTCI (°C)")
    axes[0].set_title("Predicted vs True UTCI")
    axes[0].legend()

    # Error distribution
    axes[1].hist(errors, bins=50, edgecolor="black", alpha=0.7)
    axes[1].axvline(0, color="r", linestyle="--")
    axes[1].set_xlabel("Prediction Error (°C)")
    axes[1].set_ylabel("Count")
    axes[1].set_title(
        f"Error Distribution (MAE={np.mean(np.abs(errors)):.2f}°C)",
    )

    plt.tight_layout()
    plt.show()

    # Compute metrics
    rmse = float(np.sqrt(np.mean(errors**2)))
    mae = float(np.mean(np.abs(errors)))
    r2 = float(
        1 - np.sum(errors**2) / np.sum((true - true.mean()) ** 2),
    )
    print(f"RMSE: {rmse:.2f} °C")
    print(f"MAE:  {mae:.2f} °C")
    print(f"R²:   {r2:.4f}")

    return {
        "eval_rmse_utci": rmse,
        "eval_mae_utci": mae,
        "eval_r2": r2,
    }


eval_metrics = evaluate(predictions)

# Log evaluation metrics into the same MLflow run
with mlflow.start_run(run_id=run_id):
    mlflow.log_metrics(eval_metrics)

show_mlflow_runs(cfg)

In [ ]:
def feature_importance(
    model: pl.LightningModule,
    predictions: dict[str, Any],
    feature_names: list[str],
    target_std: float,
    target_mean: float,
) -> dict[str, float]:
    """Compute and plot permutation feature importance.

    For each feature, the column is randomly shuffled and inference
    is re-run.  The importance score is
    ``MSE_shuffled / MSE_baseline`` — higher means more important.

    Args:
        model: Trained MLP.
        predictions: Dict from :func:`predict` with
            ``val_features``, ``pred_utci``, and ``true_utci``.
        feature_names: List of feature names.
        target_std: Target σ for denormalisation.
        target_mean: Target μ for denormalisation.

    Returns:
        Dict mapping feature name → MSE ratio.
    """
    val_features: torch.Tensor = predictions["val_features"]
    true_utci: np.ndarray = predictions["true_utci"]
    pred_utci: np.ndarray = predictions["pred_utci"]

    baseline_mse: float = float(np.mean((pred_utci - true_utci) ** 2))
    importance: dict[str, float] = {}

    for i, name in enumerate(feature_names):
        feat_permuted = val_features.clone()
        perm_idx = torch.randperm(len(feat_permuted))
        feat_permuted[:, i] = feat_permuted[perm_idx, i]
        with torch.no_grad():
            perm_pred = (
                model(feat_permuted).numpy().squeeze()
                * target_std
                + target_mean
            )
        mse_permuted = float(np.mean((perm_pred - true_utci) ** 2))
        importance[name] = mse_permuted / baseline_mse

    # Plot
    names = list(importance.keys())
    values = list(importance.values())
    sorted_idx = np.argsort(values)[::-1]

    plt.figure(figsize=(8, 4))
    plt.barh(
        [names[i] for i in sorted_idx],
        [values[i] for i in sorted_idx],
    )
    plt.axvline(1.0, color="r", linestyle="--", label="Baseline")
    plt.xlabel("MSE ratio (higher = more important)")
    plt.title("Permutation Feature Importance")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return importance


fi = feature_importance(
    model, predictions, data["feature_names"],
    dm.target_std, dm.target_mean,
)

## 7. Playground

Run the full pipeline in one cell.  **Change the parameters below** and re-run to experiment!

You can customise:
- **Config values** — `max_epochs`, `hidden_sizes`, `dropout`, `learning_rate`, `batch_size`
- **Time range** — `time_start` / `time_end` (ERA5 data available from `1940-01-01`)
- **Loss function** — swap `nn.MSELoss()` for `nn.L1Loss()`, `nn.HuberLoss()`, etc.
- **Optimizer** — try `torch.optim.AdamW`, `torch.optim.SGD`, etc.
- **Activation** — try `nn.GELU`, `nn.SiLU`, `nn.LeakyReLU`, etc.
- **Model class** — define your own `LightningModule` and pass it via `model_cls`

In [ ]:
# ── 1. Config (change these!) ──────────────────────────────────
run_name = "mlp_thermal_comfort"  # ← change before each run!

cfg = load_config(
    max_epochs=30,
    hidden_sizes=[64, 32, 16],
    dropout=0.1,
    learning_rate=1e-3,
    batch_size=64,
)

# ── 2. Loss function ──────────────────────────────────────────
# Try: nn.L1Loss(), nn.HuberLoss(), nn.SmoothL1Loss()
loss_fn = nn.MSELoss()

# ── 3. Optimizer ───────────────────────────────────────────────
# Try: torch.optim.SGD, torch.optim.AdamW
optimizer_cls = torch.optim.Adam

# ── 4. Activation ─────────────────────────────────────────────
# Try: nn.GELU, nn.SiLU, nn.LeakyReLU
activation_cls = nn.ReLU

# ── 5. Run pipeline ───────────────────────────────────────────
data = load_data(cfg)
plot_raw_data(data)
dm = create_datamodule(data, cfg)
model = create_model(
    cfg,
    n_features=len(data["feature_names"]),
    loss_fn=loss_fn,
    optimizer_cls=optimizer_cls,
    activation_cls=activation_cls,
)
trainer, metrics, run_id = train(
    model, dm, cfg, dm.target_std, run_name=run_name,
)
predictions = predict(model, dm)
eval_metrics = evaluate(predictions)

# ── 6. Log evaluation metrics into MLflow ──────────────────────
with mlflow.start_run(run_id=run_id):
    mlflow.log_metrics(eval_metrics)

fi = feature_importance(
    model, predictions, data["feature_names"],
    dm.target_std, dm.target_mean,
)

# ── 7. Compare experiments ────────────────────────────────────
show_mlflow_runs(cfg)

## 8. Exercises

Use the **Playground** cell above to experiment — change one parameter at a time and re-run.

1. **Thermal comfort classes**: Convert UTCI to stress categories (no stress, moderate heat stress, strong heat stress) and train a classifier
2. **Architecture search**: Try different hidden layer sizes and depths — use Hydra multirun or manual sweeps
3. **Add batch normalisation**: Insert `nn.BatchNorm1d` layers and compare performance
4. **More cities**: Add more European cities to `cfg.data.cities` and retrain — does the model generalise?
5. **More variables**: Load additional ERA5 variables from the ARCO-ERA5 store (e.g., `boundary_layer_height`, `soil_temperature_level_1`)
6. **Temporal features**: Add hour-of-day and day-of-year as cyclic features (sin/cos encoding)
7. **Compare experiments**: Call `show_mlflow_runs(cfg)` to see a summary of all your runs.  For a full UI, local users can run `!mlflow ui` and open `http://localhost:5000`